# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [3]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5051 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [1]:
import functools

# TODO: Implementacja dekoratora
def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for arg in args:
            if isinstance(arg, list):
                print(f"Długość listy: {len(arg)}")
        return func(*args, **kwargs)
    return wrapper


# Test:
@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")

process_data([1, 2, 3, 4, 5], "danych")
process_data([], "danych")
process_data("sad" ,[1, 2, 3, 4, 5])

Długość listy: 5
Przetwarzanie danych
Długość listy: 0
Przetwarzanie danych
Długość listy: 5
Przetwarzanie [1, 2, 3, 4, 5]


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [3]:
import functools
from datetime import datetime
import time
from pathlib import Path

def save_log(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
       
        if args:
            filename = args[0]
            log_dir = Path(__file__).parent / "LogsLab2"
            log_dir.mkdir(exist_ok=True)
            path = log_dir / filename
            
            with open(path, "a", encoding="utf-8") as log_file:
                log_file.write(f"{func.__name__} / {datetime.now()} / {end_time - start_time:.4f} s\n")
        return result
    return wrapper




@save_log
def logger(filename):
    # przykładowa funkcja - może wykonać dowolną pracę przed zapisem logu
    return f"Logged to {filename}"


--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [6]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [4]:
# TODO: Implementacja deskryptora logującego dostęp
class AccessLogger:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner):
        print(f"Accessing {self.name} ")
        return instance.__dict__.get(self.name, None)
    def __set__(self, instance, value):
        print(f"Setting {self.name}  to {value}")
        instance.__dict__[self.name] = value

class Uzytkownik:
    imie = AccessLogger()
    wiek = AccessLogger()

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek


u = Uzytkownik("Jan", 30)
print(u.imie)
print(u.wiek)

Setting imie  to Jan
Setting wiek  to 30
Accessing imie 
Jan
Accessing wiek 
30


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [ ]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [5]:
# TODO: Implementacja generatora ciągu Collatza

class CollatzGenerator:
    def __init__(self, n):
        self.n = n

    def __iter__(self):
        return self
    def __next__(self):
        if self.n == 1:
            raise StopIteration
        elif self.n % 2 == 0:
            self.n = self.n // 2
        else:
            self.n = 3 * self.n + 1
        return self.n
    
def collatz_generator(n):
    return CollatzGenerator(n)

# Test:
for status in collatz_generator(10):
    print(status)

5
16
8
4
2
1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [6]:
current_user = {"username": "admin", "role": "superuser"}

# TODO: Implementacja dekoratora @require_role
def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user.get("role") != role:
                raise PermissionError("You do not have the required role to access this function.")
            return func(*args, **kwargs)
        return wrapper
    return decorator    

@require_role("superuser")
def admin_function():
    print("This is an admin function.")
    pass


admin_function()

This is an admin function.


### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [7]:
# TODO: Implementacja deskryptora Typed
class Typed:
    def __set_name__(self, owner, name):
        self.name = name
        self.private_name = f"_{name}"

    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.private_name, None)

    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"Property '{self.name}' must be {self.expected_type.__name__}, "
                f"got {type(value).__name__}"
            )
        setattr(obj, self.private_name, value)


# Test:
class Produkt:
    nazwa = Typed(str)
    cena = Typed(float)

p = Produkt()
p.nazwa = "Laptop"   # OK
p.cena = 3499.99     # OK
print(p.nazwa, p.cena)

try:
    p.cena = "drogi"  # TypeError
except TypeError as e:
    print(e)

Laptop 3499.99
Property 'cena' must be float, got str


### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [8]:
# TODO: Implementacja generatora liczb pierwszych
def prime_generator():
    primes = []
    n = 2
    while True:
        if all(n % p != 0 for p in primes):
            primes.append(n)
            yield n
        n += 1

primes_ending_7 = (p for p in prime_generator() if p % 10 == 7)

for _ in range(10):
    print(next(primes_ending_7))

7
17
37
47
67
97
107
127
137
157
